# Regresi Logistik (Logistic Regression)

Course 3: Supervised Learning, Klasifikasi

Regresi logistik adalah model klasifikasi yang menghitung peluang sebuah data masuk ke tiap kelas. Model ini juga jadi dasar dari neural network. Di notebook ini aku memakainya untuk klasifikasi tiga kelas, membandingkan penalti L2, L1, dan elastic-net, lalu membaca koefisiennya.

## Dataset

Datanya `food_items.csv`, informasi gizi berbagai makanan untuk penderita diabetes. Ada 17 kolom nutrisi (kalori, lemak, protein, gula, dan seterusnya) dan kolom `class` berisi saran konsumsi: `More Often`, `In Moderation`, atau `Less Often`. Tujuannya memprediksi saran konsumsi dari kandungan gizinya. Dibaca langsung dari link:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/food_items.csv

## 1. Import dan baca data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/food_items.csv"
food = pd.read_csv(URL)
print(food.shape)
food.head(10)

In [ ]:
fitur = food.columns[:-1].tolist()
print(len(fitur), "fitur:", fitur)
food[fitur].describe().round(2)

Semua fitur numerik, jadi tidak perlu encoding. Yang perlu dilakukan hanya scaling, supaya koefisien antar fitur nanti bisa dibandingkan langsung.

## 2. Distribusi kelas

In [ ]:
dist = food["class"].value_counts()
print((food["class"].value_counts(normalize=True) * 100).round(1).to_string())
dist.plot.bar(figsize=(6, 3), color=["tab:orange", "tab:red", "tab:green"][:len(dist)])
plt.ylabel("jumlah")
plt.show()

Ada tiga kelas dan jumlahnya tidak seimbang. Karena kelasnya lebih dari dua, regresi logistik yang dipakai adalah versi **multinomial**: model menghasilkan satu distribusi peluang untuk semua kelas sekaligus (bukan satu model biner per kelas seperti skema one-vs-rest).

## 3. Persiapan data

Data dibagi 80% train dan 20% test (stratified supaya proporsi kelas terjaga). Scaler di-`fit` hanya pada data train, lalu dipakai untuk mengubah data test.

In [ ]:
X_raw = food[fitur]
le = LabelEncoder()
y = le.fit_transform(food["class"])
print("kode kelas:", dict(zip(le.classes_, le.transform(le.classes_))))

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, stratify=y, random_state=RS)

scaler = MinMaxScaler().fit(X_tr_raw)
X_train = pd.DataFrame(scaler.transform(X_tr_raw), columns=fitur, index=X_tr_raw.index)
X_test = pd.DataFrame(scaler.transform(X_te_raw), columns=fitur, index=X_te_raw.index)

print("train:", X_train.shape, "| test:", X_test.shape)
print("rentang fitur train:", X_train.values.min(), "sampai", X_train.values.max())

## 4. Fungsi evaluasi

In [ ]:
def evaluasi(y_benar, y_pred, nama_model=""):
    # tabel precision/recall/F1 per kelas + akurasi keseluruhan
    laporan = classification_report(y_benar, y_pred, target_names=le.classes_, output_dict=True, zero_division=0)
    tabel = pd.DataFrame(laporan).T.loc[list(le.classes_) + ["macro avg"], ["precision", "recall", "f1-score"]]
    print(f"{nama_model} | akurasi = {accuracy_score(y_benar, y_pred):.3f}")
    return tabel.round(3)

## 5. Model dengan penalti L2

Penalti L2 mengecilkan koefisien tapi tidak membuatnya nol. Solver `lbfgs` cocok untuk L2 dan kasus multinomial.

In [ ]:
model_l2 = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000, random_state=RS)
model_l2.fit(X_train, y_train)
pred_l2 = model_l2.predict(X_test)
evaluasi(y_test, pred_l2, "L2")

Perhatikan recall tiap kelas. Kelas yang jumlahnya sedikit biasanya lebih sulit dikenali. Ini masalah klasifikasi tidak seimbang yang dibahas di notebook tersendiri.

## 6. Model dengan penalti L1

Penalti L1 bisa membuat sebagian koefisien tepat nol, jadi sekaligus menyeleksi fitur. Untuk L1 dengan kasus multinomial dipakai solver `saga`.

In [ ]:
model_l1 = LogisticRegression(penalty="l1", solver="saga", max_iter=2000, random_state=RS)
model_l1.fit(X_train, y_train)
pred_l1 = model_l1.predict(X_test)
evaluasi(y_test, pred_l1, "L1")

### Peluang tiap kelas

`predict_proba` memberi distribusi peluang. Kelas yang diprediksi adalah yang peluangnya paling besar, sama dengan hasil `predict`.

In [ ]:
peluang = model_l1.predict_proba(X_test.iloc[[0]])[0]
print(pd.Series(peluang, index=le.classes_).round(3).to_string())
print("prediksi:", le.classes_[model_l1.predict(X_test.iloc[[0]])[0]], "| kelas sebenarnya:", le.classes_[y_test[0]])

### Perbandingan L2 dan L1

In [ ]:
banding = pd.DataFrame({
    "L2": {"akurasi": accuracy_score(y_test, pred_l2), "koefisien nol": int((model_l2.coef_ == 0).sum())},
    "L1": {"akurasi": accuracy_score(y_test, pred_l1), "koefisien nol": int((model_l1.coef_ == 0).sum())},
}).T
banding.round(3)

Penalti L1 menghasilkan lebih banyak koefisien nol, yang berarti modelnya lebih sederhana. Mana yang lebih akurat bisa dilihat dari tabel di atas.

## 7. Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_l1, display_labels=le.classes_,
                                        normalize="true", cmap="Blues", values_format=".2f", ax=ax)
ax.set_title("Model L1 (dinormalisasi per kelas sebenarnya)")
plt.xticks(rotation=20)
plt.show()

Tiap baris menunjukkan kelas sebenarnya dan angkanya adalah proporsi yang diprediksi ke masing-masing kelas. Angka di diagonal adalah recall tiap kelas.

## 8. Interpretasi koefisien

Koefisien menunjukkan arah pengaruh fitur terhadap tiap kelas. Karena fitur sudah diskalakan ke 0 sampai 1, koefisien antar fitur bisa dibandingkan. Koefisien positif menaikkan peluang kelas itu, negatif menurunkannya. Karena ada fungsi softmax, pembacaannya tidak setegas regresi linear biasa, tapi tetap memberi gambaran.

In [ ]:
koef = pd.DataFrame(model_l1.coef_, index=le.classes_, columns=fitur)

def plot_koef(model_koef, kelas, ambang=0.01):
    k = model_koef.loc[kelas]
    k = k[k.abs() >= ambang].sort_values()
    warna = ["tab:red" if v < 0 else "tab:green" for v in k]
    k.plot.barh(color=warna, figsize=(7, max(3, 0.3 * len(k))))
    plt.title(f"Koefisien untuk kelas '{kelas}'")
    plt.xlabel("koefisien")
    plt.show()

for kelas in le.classes_:
    plot_koef(koef, kelas)

In [ ]:
for kelas in le.classes_:
    k = koef.loc[kelas].sort_values()
    print(f"[{kelas}]")
    print("  paling menaikkan peluang:", ", ".join(k.index[::-1][:3]))
    print("  paling menurunkan peluang:", ", ".join(k.index[:3]))

Dari daftar di atas bisa dilihat nutrisi mana yang paling mendorong sebuah makanan masuk ke kelas tertentu, misalnya apakah nutrisi yang biasanya dianggap kurang sehat cenderung mendorong ke kelas `Less Often`.

## 9. Latihan: elastic-net

Elastic-net menggabungkan penalti L1 dan L2 dan diatur dengan `l1_ratio` (di antara 0 dan 1). Di scikit-learn, elastic-net hanya didukung solver `saga`.

In [ ]:
model_en = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=2000, random_state=RS)
model_en.fit(X_train, y_train)
pred_en = model_en.predict(X_test)
evaluasi(y_test, pred_en, "Elastic-net (l1_ratio=0.5)")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_en, display_labels=le.classes_,
                                        normalize="true", cmap="Greens", values_format=".2f", ax=ax)
ax.set_title("Elastic-net")
plt.xticks(rotation=20)
plt.show()

In [ ]:
koef_en = pd.DataFrame(model_en.coef_, index=le.classes_, columns=fitur)
plot_koef(koef_en, "Less Often")

pd.DataFrame({
    "L2": {"akurasi": accuracy_score(y_test, pred_l2), "koefisien nol": int((model_l2.coef_ == 0).sum())},
    "L1": {"akurasi": accuracy_score(y_test, pred_l1), "koefisien nol": int((model_l1.coef_ == 0).sum())},
    "Elastic-net": {"akurasi": accuracy_score(y_test, pred_en), "koefisien nol": int((model_en.coef_ == 0).sum())},
}).T.round(3)

Elastic-net berada di antara L2 dan L1: sebagian koefisien jadi nol, tapi tidak seagresif L1 murni.

## Ringkasan

- Regresi logistik multinomial menghasilkan peluang untuk tiap kelas, dan kelas dengan peluang terbesar jadi prediksi.
- Penalti L2 mengecilkan koefisien, L1 menjadikan sebagian koefisien nol (seleksi fitur), elastic-net menggabungkan keduanya.
- Untuk kelas yang jumlahnya sedikit, recall biasanya lebih rendah. Akurasi saja tidak cukup untuk menilai model di data seperti ini.
- Koefisien model memberi gambaran fitur mana yang mendorong tiap kelas, selama fiturnya sudah diskalakan.